In [ ]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

# 데이터 확인하기 2025.11.20 152개 컬럼을 사용해 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.preprocessing import StandardScaler # 데이터 전처리용

import matplotlib.pyplot as plt
import seaborn as sns

from utils.preprocessing import load_data, split_features_target
from utils.model_utils   import save_model
from utils.user_utils    import get_clf_eval

# import importlib
# importlib.reload(preprocessing)


In [6]:
# 데이터 로딩
data_path_train = '../data/train.csv'
data_path_test  = '../data/test.csv'

train, test = load_data(data_path_train, data_path_test)

In [3]:
# Data 전처리 1. zero_count_rate이 99%인 컬럼 제거하기 
remove_cols = pd.read_csv('../doc/remove_cols.csv', header=0).squeeze()
train.drop(columns=remove_cols, axis=1, inplace=True)
test.drop(columns=remove_cols, axis=1, inplace=True)

In [ ]:
# 데이터 할당
X_features, y_labels = split_features_target(train) # ID와 TARGET 모두 제거하고 X_train 만들기
X_test     = test.drop(columns=['ID'], axis=1) # test 데이터에서도 ID 제거 
X_features.columns # 152개

Index(['var3', 'var15', 'imp_ent_var16_ult1', 'imp_op_var39_comer_ult1',
       'imp_op_var39_comer_ult3', 'imp_op_var41_comer_ult1',
       'imp_op_var41_comer_ult3', 'imp_op_var41_efect_ult1',
       'imp_op_var41_efect_ult3', 'imp_op_var41_ult1',
       ...
       'saldo_medio_var8_ult3', 'saldo_medio_var12_hace2',
       'saldo_medio_var12_hace3', 'saldo_medio_var12_ult1',
       'saldo_medio_var12_ult3', 'saldo_medio_var13_corto_hace2',
       'saldo_medio_var13_corto_hace3', 'saldo_medio_var13_corto_ult1',
       'saldo_medio_var13_corto_ult3', 'var38'],
      dtype='object', length=152)

In [5]:
# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [ ]:
# 152개 컬럼의 데이터 전처리한 것 저장해 두기, 다음에 다시 테스트할때 대비
# X_features.to_csv('../data/x_train_152cols.csv')
# X_test.to_csv('../data/x_test_152cols.csv')

In [6]:
# 스케일링
# X_train_scaled, X_test_scaled, scaler = scale_data(X_train=X, X_test=X_test)
scaler        = StandardScaler()
X_scaled      = scaler.fit_transform(X_features)
X_test_scaled = scaler.transform(X_test)


In [9]:
# 레이블의 분포 확인
cust_cnt = y_labels.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [25]:
type(X_scaled)

numpy.ndarray

In [ ]:
# first testing model
# XGBoost (xgb) : yjh, kjh
# LightGBM(lgbm) : lsj, ujm
# Random Forest(rf) : lkj, kjh
# Logistic Regression(lr) : yjh, ujm


In [7]:
# 학습/테스트 데이터 분리
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
  X_features, 
  y_labels,
  test_size    = 0.2, 
  random_state = 23, # 세미프로젝트3조
  stratify = y_labels
)


In [ ]:
# Model 학습, 평가
from sklearn.ensemble import RandomForestClassifier 

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 100,
  max_depth    = 8, # RF : 애가 핵심이야 Tree 계열이니까~ 약한 Tree로 만들어야해 그래서 max_depth로 자른거
  n_jobs       = -1 # 병렬처리 여부 
)
rf_clf.fit(X_train, y_train)    # 학습
save_model(rf_clf, 'RandomForest_basic')
pred       = rf_clf.predict(X_val)   # 예측
pred_proba = rf_clf.predict_proba(X_val)[:,1] # 예측확률

get_clf_eval(y_test=y_val, pred=pred, pred_proba=pred_proba)

AUC: 0.8184, 정확도: 0.9604, 정밀도: 0.5000, 재현율: 0.0017, F1: 0.0033


In [58]:
X_train.shape

(60816, 152)